In [10]:
#pip install networkx pandas requests

In [11]:
import re
import math
import time
import copy
import json
import os
import requests
import networkx as nx
import pandas as pd
from tqdm import tqdm
from IPython.display import FileLink

In [12]:
tfl_api_key    = "a4866fe16555474daef639d13346f389"
link_freq_path = '/kaggle/input/datasets/ryanh34/numbat2024monlinkfreqdata/numbat2024MONlinkfreqdata.csv'

# candidate stations to test (lat, lon, label)
candidate_stations = [
    (51.515648, -0.401909, "Imaginary A"),
    (51.340649, -0.144658, "Imaginary B"),
    (51.547139, -0.055748, "Imaginary C"),
]

# the above are the locations kyrill recommended from his map
# conversions to british national grid (easting, northing):
# A: 510987, 180846
# B: 529328, 161807
# C: 534911, 184930

n_connections = 3    
benefit_radius_km = 2.5 # radius to find stations that benefit from the candidate
min_nearby_stations = 2.5   

# saves api results
cache_path = '/kaggle/working/journey_cache.json'

In [13]:
def strip_rail_suffix(name: str) -> str:
    # removes NR/LU tags
    return re.sub(r'\s+(NR|LU)$', '', str(name).strip(), flags=re.IGNORECASE)

def haversine(lat1, lon1, lat2, lon2):
    # straight line distance in meters between two coords
    R  = 6371000
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp = math.radians(lat2 - lat1)
    dl = math.radians(lon2 - lon1)
    a  = math.sin(dp/2)**2 + math.cos(p1)*math.cos(p2)*math.sin(dl/2)**2
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

In [14]:
# station-to-station connections we need to build the graph
lf = pd.read_csv(link_freq_path)
lf.columns = lf.columns.str.strip()

lf['From Station Clean'] = lf['From Station'].apply(strip_rail_suffix)
lf['To Station Clean']   = lf['To Station'].apply(strip_rail_suffix)
lf['Total'] = (lf['Total'].astype(str).str.replace(',', '')
                          .str.strip()
                          .pipe(pd.to_numeric, errors='coerce')
                          .fillna(0).astype(int))

print(f"rows: {len(lf)}, unique links: {lf[['From Station Clean','To Station Clean']].drop_duplicates().shape[0]}")


# sum up total flows for each station appearing on either end of a link
# this is our "population" weight for the reduction formula
from_totals = lf.groupby('From Station Clean')['Total'].sum()
to_totals   = lf.groupby('To Station Clean')['Total'].sum()
station_population = (from_totals.add(to_totals, fill_value=0)
                                 .reset_index()
                                 .rename(columns={'From Station Clean': 'Station', 0: 'Population'}))
station_population.columns = ['Station', 'Population']
print(f"unique stations: {len(station_population)}")

rows: 5599, unique links: 1087
unique stations: 477


In [15]:
def get_station_coords(station_name):
    url = f"https://api.tfl.gov.uk/StopPoint/Search/{requests.utils.quote(station_name)}?app_key={tfl_api_key}"
    try:
        r = requests.get(url, timeout=10).json()
        matches = r.get('matches', [])
        if matches:
            return (matches[0]['lat'], matches[0]['lon'])
    except Exception:
        pass
    return None

all_stations = set(lf['From Station Clean'].tolist() + lf['To Station Clean'].tolist())
station_coords = {}
for station in tqdm(all_stations, desc="station coords"):
    coords = get_station_coords(station)
    if coords:
        station_coords[station] = coords
    time.sleep(0.05)

print(f"fetched {len(station_coords)} / {len(all_stations)} coords")


station coords: 100%|██████████| 477/477 [01:46<00:00,  4.47it/s]

fetched 453 / 477 coords


In [29]:
# time saver cache
if os.path.exists(cache_path):
    with open(cache_path, 'r') as f:
        journey_cache = json.load(f)
    print(f"loaded {len(journey_cache)} cached journey times")
else:
    journey_cache = {}

def get_journey_time(from_coords, to_coords):
    # returns journey time in seconds, caches results to avoid repeat calls
    key = f"{from_coords[0]},{from_coords[1]}|{to_coords[0]},{to_coords[1]}"
    if key in journey_cache:
        return journey_cache[key]
    url = (f"https://api.tfl.gov.uk/Journey/JourneyResults/"
           f"{from_coords[0]},{from_coords[1]}/to/"
           f"{to_coords[0]},{to_coords[1]}?app_key={tfl_api_key}")
    try:
        r = requests.get(url, timeout=10).json()
        secs = r['journeys'][0]['duration'] * 60
    except Exception:
        secs = None
    journey_cache[key] = secs
    time.sleep(0.05)
    return secs

link_pairs = (lf[['From Station Clean', 'To Station Clean']]
              .drop_duplicates()
              .values.tolist())

print(f"\nfetching journey times for {len(link_pairs)} pairs...")
edge_times = {}
for frm, to in tqdm(link_pairs, desc="journey times"):
    if frm in station_coords and to in station_coords:
        t = get_journey_time(station_coords[frm], station_coords[to])
        if t:
            edge_times[(frm, to)] = t

# save cache so next run is instant
with open(cache_path, 'w') as f:
    json.dump(journey_cache, f)
print(f"cache saved ({len(journey_cache)} entries)")
print(f"edge times fetched: {len(edge_times)}")

# takes forever the first time u run it

loaded 1050 cached journey times

fetching journey times for 1087 pairs...


journey times: 100%|██████████| 1087/1087 [00:00<00:00, 239177.86it/s]

cache saved (1050 entries)
edge times fetched: 981


In [28]:
# nodes = stations, edges = links between them weighted by journey time
G = nx.DiGraph()

for station, (lat, lon) in station_coords.items():
    G.add_node(station, lat=lat, lon=lon)

for (frm, to), t in edge_times.items():
    G.add_edge(frm, to, travel_time=t)

print(f"graph: {len(G.nodes)} nodes, {len(G.edges)} edges")

# check connectivity- ideally we want 1 big connected component
components = list(nx.weakly_connected_components(G))
largest_cc  = max(components, key=len)
print(f"connected components: {len(components)}, largest: {len(largest_cc)} nodes")

# stitch smaller components into the main one so nothing is isolated
for component in components:
    if component == largest_cc:
        continue
    for station in component:
        if station not in station_coords:
            continue
        slat, slon = station_coords[station]
        # find nearest station in the main component and connect to it
        nearest = min(
            [(s, haversine(slat, slon, station_coords[s][0], station_coords[s][1]))
             for s in largest_cc if s in station_coords],
            key=lambda x: x[1]
        )
        nb, dist = nearest
        t = get_journey_time((slat, slon), (station_coords[nb][0], station_coords[nb][1]))
        t = t or dist / 11.1  # fallback to haversine estimate if api fails
        G.add_edge(station, nb, travel_time=t)
        G.add_edge(nb, station, travel_time=t)

print(f"after merge: {len(G.nodes)} nodes, {len(G.edges)} edges")
print(f"components after merge: {nx.number_weakly_connected_components(G)}")

graph: 453 nodes, 981 edges
connected components: 8, largest: 402 nodes
after merge: 453 nodes, 1083 edges
components after merge: 1


In [23]:
def get_nearby_stations(G_base, lat, lon, radius_km, min_stations=min_nearby_stations):
    # expands radius until we find enough nearby stations
    r = radius_km
    while True:
        nearby = [
            s for s, d in G_base.nodes(data=True)
            if haversine(lat, lon, d['lat'], d['lon']) / 1000 <= r
        ]
        if len(nearby) >= min_stations or r > 20:
            if r > radius_km:
                print(f"  (expanded radius to {r:.1f}km to find {len(nearby)} nearby stations)")
            return nearby
        r += 0.5

def add_candidate_to_graph(G_base, lat, lon, label, n_connections):
    # adds the candidate station and connects it to its n nearest neighbours
    G_new = copy.deepcopy(G_base)
    G_new.add_node(label, lat=lat, lon=lon)

    distances = sorted(
        [(s, haversine(lat, lon, d['lat'], d['lon']))
         for s, d in G_new.nodes(data=True) if s != label],
        key=lambda x: x[1]
    )
    nearest_n = [s for s, _ in distances[:n_connections]]

    print(f"  connecting {label} to: {nearest_n}")
    for nb in nearest_n:
        nb_lat, nb_lon = G_new.nodes[nb]['lat'], G_new.nodes[nb]['lon']
        t_to   = get_journey_time((lat, lon), (nb_lat, nb_lon))
        t_from = get_journey_time((nb_lat, nb_lon), (lat, lon))
        # save cache after each candidate in case something crashes
        with open(cache_path, 'w') as f:
            json.dump(journey_cache, f)
        t_to   = t_to   or haversine(lat, lon, nb_lat, nb_lon) / 11.1  # fallback
        t_from = t_from or haversine(nb_lat, nb_lon, lat, lon) / 11.1
        print(f"    - {nb}: to={t_to:.0f}s, from={t_from:.0f}s")
        G_new.add_edge(label, nb, travel_time=t_to)
        G_new.add_edge(nb, label, travel_time=t_from)

    return G_new

def compute_total_reduction(G_base, G_new, station_population, candidate_lat, candidate_lon, radius_km):
    # total reduction = sum of (population * travel time saved) for all nearby origin stations
    # only looks at stations within the benefit radius — those are the ones who'd actually use it
    pop_dict = dict(zip(station_population['Station'], station_population['Population']))
    nearby   = get_nearby_stations(G_base, candidate_lat, candidate_lon, radius_km)
    origins  = [s for s in nearby if s in pop_dict and s in G_base.nodes]

    print(f"  nearby stations within {radius_km}km: {origins}")

    total_reduction = 0.0
    breakdown = []

    for origin in origins:
        pop   = pop_dict[origin]
        d_old = nx.single_source_dijkstra_path_length(G_base, source=origin, weight="travel_time")
        d_new = nx.single_source_dijkstra_path_length(G_new,  source=origin, weight="travel_time")

        for dest in d_old:
            if dest == origin:
                continue
            old_t     = d_old[dest]
            new_t     = d_new.get(dest, old_t)
            reduction = old_t - new_t
            if reduction > 0:
                total_reduction += pop * reduction
                breakdown.append({
                    'origin': origin, 'dest': dest,
                    'population': pop,
                    'd_old_s': old_t, 'd_new_s': new_t,
                    'reduction_s': reduction,
                    'weighted': pop * reduction
                })

    return total_reduction, pd.DataFrame(breakdown)

# results for each candidate
results = []

for lat, lon, label in candidate_stations:
    print(f"\nevaluating: {label} ({lat}, {lon})")
    G_new = add_candidate_to_graph(G, lat, lon, label, n_connections)
    total_reduction, breakdown_df = compute_total_reduction(
        G, G_new, station_population, lat, lon, benefit_radius_km
    )
    print(f"    - total reduction: {total_reduction:,.0f} person-seconds")
    results.append({'label': label, 'lat': lat, 'lon': lon, 'total_reduction': total_reduction})


evaluating: Imaginary A (51.515648, -0.401909)
  connecting Imaginary A to: ['Hayes & Harlington', 'Southall', 'Northolt']
    - Hayes & Harlington: to=1200s, from=1260s
    - Southall: to=1320s, from=1980s
    - Northolt: to=1800s, from=1560s
  (expanded radius to 4.5km to find 4 nearby stations)
  nearby stations within 2.5km: ['Hayes & Harlington', 'Southall', 'Hanwell', 'Northolt']
    - total reduction: 47,554,080 person-seconds

evaluating: Imaginary B (51.340649, -0.144658)
  connecting Imaginary B to: ['Wandle Park', 'Waddon Marsh', 'Church Street']
    - Wandle Park: to=2280s, from=1740s
    - Waddon Marsh: to=2100s, from=2340s
    - Church Street: to=2760s, from=1860s
  (expanded radius to 5.0km to find 7 nearby stations)
  nearby stations within 2.5km: ['Church Street', 'Reeves Corner', 'Wandle Park', 'Centrale', 'Ampere Way', 'George Street', 'Waddon Marsh']
    - total reduction: 0 person-seconds

evaluating: Imaginary C (51.547139, -0.055748)
  connecting Imaginary C to:

In [26]:
results_df = pd.DataFrame(results).sort_values('total_reduction', ascending=False)
print("\ncandidate station ranking:")
print(results_df.to_string(index=False))
results_df.to_csv("candidate_reduction_scores.csv", index=False)


candidate station ranking:
      label       lat       lon  total_reduction
Imaginary C 51.547139 -0.055748     1.376829e+09
Imaginary A 51.515648 -0.401909     4.755408e+07
Imaginary B 51.340649 -0.144658     0.000000e+00


In [24]:
FileLink('candidate_reduction_scores.csv')

/kaggle/working/candidate_reduction_scores.csv

In [25]:
FileLink('journey_cache.json')

/kaggle/working/journey_cache.json

# Results Discussion

Using Kyrylo's location points:

Imaginary C (at abt 1.1 billion person-seconds) is the best with the given points. It's sitting in like the Hackney Central area, and with 10 nearby stations all feeding into it, the ripple effect across the network is way bigger than A.

Imaginary A (at abt 240 million person-seconds) is alright. Still in the Southall/Hayes area out west- it plugs a gap between busy stations and creating shortcuts that didn't previously exist.

Imaginary B (at 0) does nothing here. The nearby stations are all part of the Croydon Tramlink which is basically its own little bubble, adding a station there doesn't help anyone get anywhere new faster.

(btw person-seconds is just the unit of how many people benefit and how much time they each save combined)